# Stage 2.1b - Encoder local/global feature audit

This evaluation-only notebook tests the selected fold encoder without updating it.

- Local evidence: frozen L0/L1 features feed disposable 1x1 four-bone probes and photometric spatial-invariance tests.
- Global evidence: frozen L2/L3 descriptors feed bidirectional AP/LAT retrieval, permutation tests, and non-collapse statistics.
- Certified 3D targets create evaluation silhouettes only; they never enter FCMAE training, masking, sampling, or checkpoint selection.
- Representation underperformance is a visible WARN. Provenance failure, test access, non-finite/collapsed features, or encoder mutation is a hard failure.

Set `RUN_AUDIT=True` only after the requested `pinned_init`, `P1`, or `P2` checkpoint is available.

## What does accurate feature extraction mean here?

There is no single ground-truth feature tensor. Accuracy is therefore tested through frozen, task-relevant evidence rather than by visually inspecting activations alone.

**Local features (L0/L1).** A disposable 1x1 linear probe asks whether spatial bone information is already accessible without a nonlinear decoder. Comparison with constant-prevalence and mean-atlas controls separates encoder information from dataset-average anatomy. Clean-versus-photometric spatial cosine similarity checks that exposure changes do not move the representation geometrically.

**Global features (L2/L3).** Mean/max-pooled descriptors are evaluated by bidirectional AP-to-LAT retrieval. Recall, MRR, paired cosine margin, and permutation testing ask whether two views of the same knee are more compatible than unrelated views.

**Non-collapse evidence.** Feature standard deviation, covariance concentration, and effective rank reveal constant or near-constant representations that could otherwise produce misleading probe results.

## Why targets are allowed only in this notebook

The four-bone 3D targets are projected with the same AP/LAT geometry solely to score frozen representations. They cannot influence FCMAE masking, sampling, loss, early stopping, or checkpoint selection, preserving the self-supervised training claim.

## Interpreting PASS and WARN

A quality `WARN` means the encoder evidence is weak or regressed and must be reported; it does not automatically block the next phase. A structural failure (wrong provenance, test access, encoder mutation, non-finite values, or collapse) invalidates the audit and stops progression. Running `pinned_init`, then `P1`, then `P2` makes adaptation gains or regressions explicit.

In [1]:
import hashlib
import json
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

AUDIT_SCHEMA = "encoder_feature_audit_v1"
AUDIT_STAGE = "P1"  # pinned_init | P1 | P2
AUDIT_FOLD = 0
RUN_AUDIT = False
PROBE_EPOCHS = 100
PROBE_BATCH = 8
PROBE_LR = 1e-3
PROBE_WEIGHT_DECAY = 1e-4
BOOTSTRAPS = 2000
PERMUTATIONS = 1000
LOCAL_REGRESSION_TOLERANCE = 0.02
BONES = ["femur", "tibia", "patella", "fibula"]
PROJECTION_GEOMETRY = {
    "sdd": 1000.0, "sod": 850.0, "height": 256, "width": 256, "delx": 1.4, "dely": 1.4,
    "parameterization": "euler_angles", "convention": "ZXY", "degrees": True,
    "views": {
        "ap": {"rotation": [0.0, 0.0, 0.0], "translation": [0.0, 850.0, 0.0]},
        "lat": {"rotation": [90.0, 0.0, 0.0], "translation": [0.0, 850.0, 0.0]},
    },
}

def locate_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "configs" / "baseline_protocol_v1.json").is_file():
            return candidate
    raise FileNotFoundError("project root was not found")

ROOT = locate_root(Path.cwd())
encoder_candidates = [
    ROOT / "notebooks" / "modeling" / "01_encoder_pipeline.ipynb",
    ROOT / "notebooks" / "01_encoder_pipeline.ipynb",
    ROOT / "HPC" / "HPC_notebooks" / "modeling_HPC" / "01_encoder_pipeline.ipynb",
]
ENCODER_NOTEBOOK = next((path for path in encoder_candidates if path.is_file()), None)
if ENCODER_NOTEBOOK is None:
    raise FileNotFoundError("authoritative 01_encoder_pipeline.ipynb was not found")

source_notebook = json.loads(ENCODER_NOTEBOOK.read_text(encoding="utf-8"))
required_cells = {"01_encoder_pipeline-01", "01_encoder_pipeline-02", "01_encoder_pipeline-03", "01_encoder_pipeline-04"}
available_cells = {cell.get("id") for cell in source_notebook["cells"]}
if not required_cells.issubset(available_cells):
    raise RuntimeError("authoritative encoder notebook cell contract changed")
for cell in source_notebook["cells"]:
    if cell.get("id") in required_cells:
        exec(compile("".join(cell["source"]), f"{ENCODER_NOTEBOOK.name}:{cell['id']}", "exec"), globals())

FOLD = AUDIT_FOLD
ARTIFACT_ROOT = ROOT / "models" / STAGE2_SCHEMA / f"fold_{FOLD}"
AUDIT_ROOT = ARTIFACT_ROOT / "feature_audit" / AUDIT_STAGE.lower()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert AUDIT_STAGE in {"pinned_init", "P1", "P2"}
print({"audit_stage": AUDIT_STAGE, "fold": FOLD, "device": str(DEVICE), "run_audit": RUN_AUDIT})


{'fold': 0, 'device': 'cpu', 'run_real_data': False, 'peak_lr': 7.5e-05}
project root: C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject
{'audit_stage': 'P1', 'fold': 0, 'device': 'cpu', 'run_audit': False}


In [2]:
def tensor_sha256(array: np.ndarray) -> str:
    array = np.ascontiguousarray(array)
    h = hashlib.sha256()
    h.update(str(array.dtype).encode("ascii"))
    h.update(np.asarray(array.shape, dtype=np.int64).tobytes())
    h.update(array.tobytes())
    return h.hexdigest()

def load_audit_encoder():
    require_stage1_pass()
    if AUDIT_STAGE == "pinned_init":
        backbone = build_backbone(pretrained=True)
        checkpoint_path = PRETRAINED_CHECKPOINT_PATH
    else:
        filename = "fcmae_p1_encoder.pth" if AUDIT_STAGE == "P1" else "fcmae_p2_encoder.pth"
        checkpoint_path = ARTIFACT_ROOT / filename
        if not checkpoint_path.is_file():
            raise FileNotFoundError(f"required {AUDIT_STAGE} export is missing: {checkpoint_path}")
        payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        expected_stage = "FCMAE_P1" if AUDIT_STAGE == "P1" else "cross_view_P2"
        if payload.get("fold") != FOLD or payload.get("stage") != expected_stage:
            raise RuntimeError("audit checkpoint fold/stage mismatch")
        manifest_sha = json.loads(MANIFEST_META_PATH.read_text(encoding="utf-8"))["sha256"]
        if payload.get("manifest_sha256") != manifest_sha:
            raise RuntimeError("audit checkpoint manifest mismatch")
        backbone = build_backbone(pretrained=False)
        strict_load(backbone, payload["encoder_state"], f"{AUDIT_STAGE} audit backbone")
    encoder = FCMAEEncoder(backbone).to(DEVICE).eval()
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    return encoder, backbone.pretrained_cfg["mean"], backbone.pretrained_cfg["std"], checkpoint_path

def target_file(row, bone: str) -> Path:
    path = ROOT / row.target_path / f"{row.sample_id}_{bone}.nii.gz"
    if not path.is_file():
        raise FileNotFoundError(f"certified audit target is missing: {path}")
    return path

def render_silhouette(target_path: Path, view: str) -> np.ndarray:
    """Project one binary bone channel with the Stage 1 DRR geometry; no target enters the encoder."""
    from diffdrr.data import read
    from diffdrr.drr import DRR
    subject = read(target_path, orientation="AP", bone_attenuation_multiplier=1.0)
    renderer = DRR(subject, sdd=PROJECTION_GEOMETRY["sdd"], height=256, width=256,
                   delx=PROJECTION_GEOMETRY["delx"], dely=PROJECTION_GEOMETRY["dely"]).to(DEVICE)
    cfg = PROJECTION_GEOMETRY["views"][view]
    rotation = torch.tensor([cfg["rotation"]], dtype=torch.float32, device=DEVICE)
    translation = torch.tensor([cfg["translation"]], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        line_integral = renderer(rotation, translation, parameterization="euler_angles",
                                 convention="ZXY", degrees=True)[0, 0]
    silhouette = (line_integral.abs() > 1e-8).to(torch.uint8).cpu().numpy()
    if silhouette.shape != (256, 256):
        raise RuntimeError(f"projected target shape mismatch: {silhouette.shape}")
    return silhouette

def cache_silhouettes(rows: pd.DataFrame):
    cache_root = AUDIT_ROOT / "silhouettes"
    cache_root.mkdir(parents=True, exist_ok=True)
    records = []
    for row in rows.itertuples(index=False):
        for view in ("ap", "lat"):
            output = cache_root / f"{row.sample_id}_{view}.npz"
            if not output.is_file():
                channels = np.stack([render_silhouette(target_file(row, bone), view) for bone in BONES])
                np.savez_compressed(output, target=channels)
            target = np.load(output, allow_pickle=False)["target"].astype(np.uint8)
            if target.shape != (4, 256, 256) or not set(np.unique(target)).issubset({0, 1}):
                raise RuntimeError(f"invalid audit silhouette cache: {output}")
            records.append({"sample_id": row.sample_id, "subject_id": row.subject_id, "split": row.split,
                            "view": view, "path": str(output), "sha256": tensor_sha256(target)})
    return records


In [9]:
def feature_path(split: str, sample_id: str, view: str) -> Path:
    return AUDIT_ROOT / "features" / split / f"{sample_id}_{view}.npz"

def cache_clean_features(encoder, mean, std, rows: pd.DataFrame):
    """Cache hashable clean L0-L3 tensors; augmented L0/L1 tensors exist only for invariance scoring."""
    records = []
    expected = [[96, 64, 64], [192, 32, 32], [384, 16, 16], [768, 8, 8]]
    for row in rows.itertuples(index=False):
        for view, field in (("ap", "ap_drr_path"), ("lat", "lat_drr_path")):
            output = feature_path(row.split, row.sample_id, view)
            output.parent.mkdir(parents=True, exist_ok=True)
            if not output.is_file():
                clean = read_clean_drr(ROOT / getattr(row, field))
                raw = torch.from_numpy(clean).unsqueeze(0).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    levels = [x.float().cpu().numpy()[0] for x in encoder.extract_pyramid(encoder_input(raw, mean, std))]
                arrays = {f"L{i}": x for i, x in enumerate(levels)}
                if row.split == "validation":
                    augmented = augment_drr(clean, row.sample_id, view, 0, 0)
                    aug_raw = torch.from_numpy(augmented).unsqueeze(0).unsqueeze(0).to(DEVICE)
                    with torch.no_grad():
                        aug = encoder.extract_pyramid(encoder_input(aug_raw, mean, std))
                    arrays.update({f"aug_L{i}": aug[i].float().cpu().numpy()[0] for i in (0, 1)})
                np.savez_compressed(output, **arrays)
            payload = np.load(output, allow_pickle=False)
            shapes = [list(payload[f"L{i}"].shape) for i in range(4)]
            if shapes != expected:
                raise RuntimeError(f"feature pyramid shape mismatch for {output}: {shapes}")
            hashes, deviations = {}, {}
            for index in range(4):
                value = payload[f"L{index}"]
                if not np.isfinite(value).all():
                    raise FloatingPointError(f"non-finite clean feature: {output} L{index}")
                hashes[f"L{index}"] = tensor_sha256(value)
                deviations[f"L{index}"] = float(value.std())
                if deviations[f"L{index}"] <= FEATURE_STD_MIN:
                    raise RuntimeError(f"collapsed clean feature: {output} L{index}")
            records.append({"sample_id": row.sample_id, "subject_id": row.subject_id,
                            "fracture_status": row.fracture_status, "split": row.split, "view": view,
                            "drr_path": str(ROOT / getattr(row, field)), "feature_path": str(output),
                            "clean_feature_sha256": hashes, "feature_std": deviations})
    return records

def hard_dice_iou(probability, target):
    prediction, truth = probability >= 0.5, target >= 0.5
    intersection = (prediction & truth).sum(axis=(-2, -1)).astype(np.float64)
    pred_sum = prediction.sum(axis=(-2, -1)).astype(np.float64)
    target_sum = truth.sum(axis=(-2, -1)).astype(np.float64)
    dice_den = pred_sum + target_sum
    union = pred_sum + target_sum - intersection
    dice = np.where(dice_den == 0, 1.0, 2.0 * intersection / np.maximum(dice_den, 1.0))
    iou = np.where(union == 0, 1.0, intersection / np.maximum(union, 1.0))
    return dice, iou

def subject_bootstrap_lower(values_by_subject: dict, label: str):
    keys = sorted(values_by_subject)
    if not keys: return float("nan")
    rng = np.random.default_rng(stable_seed(SEED, FOLD, AUDIT_STAGE, label))
    draws = [np.mean([values_by_subject[key] for key in rng.choice(keys, len(keys), replace=True)])
             for _ in range(BOOTSTRAPS)]
    return float(np.quantile(draws, 0.025))


In [10]:
def load_probe_arrays(feature_records, silhouette_records, level):
    targets = {(r["sample_id"], r["view"]): r["path"] for r in silhouette_records}
    features, masks = [], []
    for record in feature_records:
        features.append(np.load(record["feature_path"], allow_pickle=False)[level].astype(np.float32))
        path = targets[(record["sample_id"], record["view"])]
        masks.append(np.load(path, allow_pickle=False)["target"].astype(np.float32))
    return np.stack(features), np.stack(masks)

def probe_loss(logits, target):
    bce = F.binary_cross_entropy_with_logits(logits, target)
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum(dim=(2, 3))
    denominator = probability.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return 0.5 * bce + 0.5 * (1.0 - ((2 * intersection + 1e-6) / (denominator + 1e-6)).mean())

def train_linear_probe(level, train_records, validation_records, silhouettes):
    """Measure linearly accessible bone location while keeping every encoder parameter frozen."""
    train_x, train_y = load_probe_arrays(train_records, silhouettes, level)
    val_x, val_y = load_probe_arrays(validation_records, silhouettes, level)
    seed_everything(SEED)
    probe = nn.Conv2d(train_x.shape[1], 4, 1).to(DEVICE)
    optimizer = torch.optim.AdamW(probe.parameters(), lr=PROBE_LR, weight_decay=PROBE_WEIGHT_DECAY)
    loader = DataLoader(TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y)),
                        batch_size=PROBE_BATCH, shuffle=True,
                        generator=torch.Generator(device="cpu").manual_seed(stable_seed(SEED, FOLD, AUDIT_STAGE, level, "probe")))
    history = []
    for epoch in range(PROBE_EPOCHS):
        probe.train(); total = count = 0
        for feature, target in loader:
            feature, target = feature.to(DEVICE), target.to(DEVICE)
            logits = F.interpolate(probe(feature), (256, 256), mode="bilinear", align_corners=False)
            loss = probe_loss(logits, target)
            if not torch.isfinite(loss): raise FloatingPointError(f"non-finite {level} probe loss")
            optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
            total += float(loss) * feature.shape[0]; count += feature.shape[0]
        history.append({"epoch": epoch, "loss": total / count})
    pd.DataFrame(history).to_csv(AUDIT_ROOT / f"{level}_probe_history.csv", index=False)
    torch.save({"evaluation_only": True, "level": level, "state": probe.state_dict()},
               AUDIT_ROOT / f"{level}_linear_probe.pth")

    probe.eval(); probabilities = []
    with torch.no_grad():
        for start in range(0, len(val_x), PROBE_BATCH):
            feature = torch.from_numpy(val_x[start:start + PROBE_BATCH]).to(DEVICE)
            logits = F.interpolate(probe(feature), (256, 256), mode="bilinear", align_corners=False)
            probabilities.append(torch.sigmoid(logits).cpu().numpy())
    probabilities = np.concatenate(probabilities)
    dice, iou = hard_dice_iou(probabilities, val_y)
    prevalence = train_y.mean(axis=(0, 2, 3), keepdims=True)
    constant_dice = hard_dice_iou(np.broadcast_to(prevalence, val_y.shape), val_y)[0].mean(axis=1)
    atlas = np.broadcast_to(train_y.mean(axis=0, keepdims=True), val_y.shape)
    atlas_dice = hard_dice_iou(atlas, val_y)[0].mean(axis=1)
    control = constant_dice if constant_dice.mean() >= atlas_dice.mean() else atlas_dice
    rows, deltas = [], {}
    for index, meta in enumerate(validation_records):
        macro = float(dice[index].mean())
        rows.append({"level": level, "sample_id": meta["sample_id"], "subject_id": meta["subject_id"],
                     "view": meta["view"], "macro_dice": macro, "macro_iou": float(iou[index].mean()),
                     **{f"{bone}_dice": float(dice[index, b]) for b, bone in enumerate(BONES)},
                     **{f"{bone}_iou": float(iou[index, b]) for b, bone in enumerate(BONES)},
                     "control_macro_dice": float(control[index])})
        deltas.setdefault(meta["subject_id"], []).append(macro - float(control[index]))
    deltas = {key: float(np.mean(value)) for key, value in deltas.items()}
    summary = {"validation_macro_dice": float(dice.mean()), "validation_macro_iou": float(iou.mean()),
               "constant_macro_dice": float(constant_dice.mean()), "atlas_macro_dice": float(atlas_dice.mean()),
               "control_delta": float(np.mean(list(deltas.values()))),
               "control_delta_ci95_low": subject_bootstrap_lower(deltas, f"{level}_probe_delta")}
    return probe, probabilities, val_y, rows, summary


In [11]:
def photometric_invariance(validation_records, level):
    """Compare same-position similarity with a spatially shuffled negative control."""
    values, grouped = [], {}
    rng = np.random.default_rng(stable_seed(SEED, FOLD, AUDIT_STAGE, level, "spatial_shuffle"))
    for record in validation_records:
        payload = np.load(record["feature_path"], allow_pickle=False)
        clean = payload[level].reshape(payload[level].shape[0], -1).T
        augmented = payload[f"aug_{level}"].reshape(payload[level].shape[0], -1).T
        clean /= np.maximum(np.linalg.norm(clean, axis=1, keepdims=True), 1e-8)
        augmented /= np.maximum(np.linalg.norm(augmented, axis=1, keepdims=True), 1e-8)
        same = float(np.mean(np.sum(clean * augmented, axis=1)))
        shuffled = float(np.mean(np.sum(clean * augmented[rng.permutation(len(augmented))], axis=1)))
        margin = same - shuffled
        values.append((same, shuffled, margin))
        grouped.setdefault(record["subject_id"], []).append(margin)
    grouped = {key: float(np.mean(value)) for key, value in grouped.items()}
    return {"same_cosine": float(np.mean([x[0] for x in values])),
            "shuffled_cosine": float(np.mean([x[1] for x in values])),
            "margin": float(np.mean([x[2] for x in values])),
            "margin_ci95_low": subject_bootstrap_lower(grouped, f"{level}_invariance")}

def descriptor(payload):
    parts = []
    for level in ("L2", "L3"):
        value = payload[level]
        vector = np.concatenate([value.mean(axis=(1, 2)), value.max(axis=(1, 2))])
        parts.append(vector / max(np.linalg.norm(vector), 1e-8))
    vector = np.concatenate(parts)
    return vector / max(np.linalg.norm(vector), 1e-8)

def retrieval_ranks(matrix, correct):
    return np.asarray([int(np.where(np.argsort(-matrix[i]) == correct[i])[0][0]) + 1 for i in range(len(matrix))])


def retrieval_evidence(validation_records):
    """Test whether clean AP/LAT descriptors retrieve the matching knee above permutation chance."""
    pairs = {}
    for record in validation_records: pairs.setdefault(record["sample_id"], {})[record["view"]] = record
    sample_ids = sorted(pairs)
    if any(set(pairs[sample]) != {"ap", "lat"} for sample in sample_ids):
        raise RuntimeError("global audit requires complete validation AP/LAT pairs")
    ap = np.stack([descriptor(np.load(pairs[s]["ap"]["feature_path"], allow_pickle=False)) for s in sample_ids])
    lat = np.stack([descriptor(np.load(pairs[s]["lat"]["feature_path"], allow_pickle=False)) for s in sample_ids])
    similarity = ap @ lat.T
    identity = np.arange(len(sample_ids))
    all_ranks = np.concatenate([retrieval_ranks(similarity, identity), retrieval_ranks(similarity.T, identity)])
    observed_mrr = float(np.mean(1.0 / all_ranks))
    rng = np.random.default_rng(stable_seed(SEED, FOLD, AUDIT_STAGE, "retrieval_permutations"))
    null = []
    for _ in range(PERMUTATIONS):
        permutation = rng.permutation(len(sample_ids)); inverse = np.argsort(permutation)
        null_ranks = np.concatenate([retrieval_ranks(similarity, permutation), retrieval_ranks(similarity.T, inverse)])
        null.append(float(np.mean(1.0 / null_ranks)))
    p_value = float((1 + np.sum(np.asarray(null) >= observed_mrr)) / (PERMUTATIONS + 1))
    margins = {}
    for index, sample_id in enumerate(sample_ids):
        off = np.concatenate([np.delete(similarity[index], index), np.delete(similarity[:, index], index)])
        subject = pairs[sample_id]["ap"]["subject_id"]
        margins.setdefault(subject, []).append(float(similarity[index, index] - off.mean()))
    margins = {key: float(np.mean(value)) for key, value in margins.items()}
    pd.DataFrame(similarity, index=sample_ids, columns=sample_ids).to_csv(AUDIT_ROOT / "ap_lat_similarity.csv")
    fig, axis = plt.subplots(figsize=(8, 7)); image = axis.imshow(similarity, cmap="viridis", vmin=-1, vmax=1)
    axis.set(title=f"{AUDIT_STAGE} AP/LAT global similarity", xlabel="LAT", ylabel="AP")
    fig.colorbar(image, ax=axis); fig.tight_layout()
    fig.savefig(AUDIT_ROOT / "ap_lat_similarity.png", dpi=170); plt.close(fig)
    return {"n_pairs": len(sample_ids), "recall_at_1": float(np.mean(all_ranks <= 1)),
            "recall_at_5": float(np.mean(all_ranks <= 5)), "mrr": observed_mrr,
            "median_rank": float(np.median(all_ranks)), "permutation_p_value": p_value,
            "paired_cosine_margin": float(np.mean(list(margins.values()))),
            "paired_margin_ci95_low": subject_bootstrap_lower(margins, "retrieval_margin")}


In [12]:
def feature_statistics(validation_records):
    """Report scale and spectrum diagnostics that expose constant or low-rank feature collapse."""
    results = {}
    for level in ("L0", "L1", "L2", "L3"):
        vectors = [np.load(r["feature_path"], allow_pickle=False)[level].mean(axis=(1, 2))
                   for r in validation_records]
        matrix = np.stack(vectors).astype(np.float64)
        centered = matrix - matrix.mean(axis=0, keepdims=True)
        singular = np.linalg.svd(centered, compute_uv=False) ** 2
        probability = singular / max(singular.sum(), 1e-12)
        nonzero = probability[probability > 0]
        effective_rank = float(np.exp(-np.sum(nonzero * np.log(nonzero))))
        results[level] = {"aggregate_std": float(matrix.std()), "effective_rank": effective_rank,
                          "largest_covariance_fraction": float(probability[0]) if len(probability) else 0.0}
    return results

def save_local_visuals(validation_records, probabilities, targets):
    selected = []
    for status in ("healthy", "fractured"):
        candidates = sorted([r for r in validation_records if r["fracture_status"] == status and r["view"] == "ap"],
                            key=lambda item: item["sample_id"])
        if candidates: selected.append(candidates[0])
    for record in selected:
        index = validation_records.index(record)
        payload = np.load(record["feature_path"], allow_pickle=False)
        fig, axes = plt.subplots(2, 2, figsize=(9, 8))
        for row, level in enumerate(("L0", "L1")):
            feature = payload[level]
            pixels = feature.reshape(feature.shape[0], -1).T
            pixels -= pixels.mean(axis=0, keepdims=True)
            _, _, vt = np.linalg.svd(pixels, full_matrices=False)
            rgb = (pixels @ vt[:3].T).reshape(feature.shape[1], feature.shape[2], 3)
            lo, hi = rgb.min(axis=(0, 1), keepdims=True), rgb.max(axis=(0, 1), keepdims=True)
            rgb = (rgb - lo) / np.maximum(hi - lo, 1e-8)
            axes[row, 0].imshow(rgb); axes[row, 0].set_title(f"{level} PCA RGB")
            axes[row, 1].imshow(np.linalg.norm(feature, axis=0), cmap="magma")
            axes[row, 1].set_title(f"{level} channel energy")
            for axis in axes[row]: axis.axis("off")
        fig.suptitle(f"{AUDIT_STAGE} {record['sample_id']} clean AP features"); fig.tight_layout()
        fig.savefig(AUDIT_ROOT / f"{record['sample_id']}_pca_energy.png", dpi=170); plt.close(fig)

        drr = read_clean_drr(Path(record["drr_path"]))
        for level in ("L0", "L1"):
            fig, axes = plt.subplots(1, 4, figsize=(16, 4))
            for bone_index, bone in enumerate(BONES):
                axes[bone_index].imshow(drr, cmap="gray", vmin=0, vmax=1)
                axes[bone_index].imshow(probabilities[level][index, bone_index], cmap="magma",
                                        alpha=0.55, vmin=0, vmax=1)
                axes[bone_index].contour(targets[level][index, bone_index], levels=[0.5],
                                         colors="cyan", linewidths=0.8)
                axes[bone_index].set_title(f"{level} {bone}"); axes[bone_index].axis("off")
            fig.tight_layout()
            fig.savefig(AUDIT_ROOT / f"{record['sample_id']}_{level}_probe_overlay.png", dpi=170)
            plt.close(fig)

def p2_preservation(local_summaries, global_summary):
    if AUDIT_STAGE != "P2": return {"status": "NOT_APPLICABLE"}
    path = ARTIFACT_ROOT / "feature_audit" / "p1" / "feature_audit_summary.json"
    if not path.is_file(): return {"status": "WARN", "reason": "P1 audit summary is missing"}
    p1 = json.loads(path.read_text(encoding="utf-8"))
    p1_local = max(value["validation_macro_dice"] for value in p1["local"]["probes"].values())
    p2_local = max(value["validation_macro_dice"] for value in local_summaries.values())
    local_drop = p1_local - p2_local
    mrr_change = global_summary["mrr"] - p1["global"]["mrr"]
    status = "PASS" if local_drop <= LOCAL_REGRESSION_TOLERANCE and mrr_change > 0 else "WARN"
    return {"status": status, "local_macro_dice_drop": local_drop, "global_mrr_change": mrr_change}


def stage_comparison(local_summaries, global_summary):
    if AUDIT_STAGE == "pinned_init": return {"status": "REFERENCE"}
    current_local = max(value["validation_macro_dice"] for value in local_summaries.values())
    prior_stages = ["pinned_init"] + (["p1"] if AUDIT_STAGE == "P2" else [])
    comparisons, missing = {}, []
    for prior_stage in prior_stages:
        path = ARTIFACT_ROOT / "feature_audit" / prior_stage / "feature_audit_summary.json"
        if not path.is_file(): missing.append(prior_stage); continue
        prior = json.loads(path.read_text(encoding="utf-8"))
        prior_local = max(value["validation_macro_dice"] for value in prior["local"]["probes"].values())
        comparisons[prior_stage] = {"local_macro_dice_change": current_local - prior_local,
                                    "global_mrr_change": global_summary["mrr"] - prior["global"]["mrr"]}
    status = "COMPLETE" if not missing else "INCOMPLETE"
    return {"status": status, "missing": missing, "comparisons": comparisons}

In [13]:
def run_feature_audit():
    """Run structural hard gates first, then emit non-blocking local/global quality evidence."""
    started = time.time(); AUDIT_ROOT.mkdir(parents=True, exist_ok=True)
    _, train_rows, validation_rows, test_rows, manifest_meta = load_certified_folds(FOLD)
    if AUDIT_FOLD == 0 and (len(train_rows), len(validation_rows), len(test_rows)) != (42, 14, 15):
        raise RuntimeError("fold-0 audit split count mismatch")
    encoder, mean, std, checkpoint_path = load_audit_encoder()
    before_hash = tensor_state_sha256(encoder.state_dict())
    audit_rows = pd.concat([train_rows, validation_rows], ignore_index=True)
    silhouettes = cache_silhouettes(audit_rows)
    feature_records = cache_clean_features(encoder, mean, std, audit_rows)
    train_features = [r for r in feature_records if r["split"] == "train"]
    validation_features = [r for r in feature_records if r["split"] == "validation"]

    probes, probabilities, targets = {}, {}, {}
    local_rows, local_summaries = [], {}
    for level in ("L0", "L1"):
        probe, probability, truth, rows, summary = train_linear_probe(
            level, train_features, validation_features, silhouettes)
        probes[level], probabilities[level], targets[level] = probe, probability, truth
        local_rows.extend(rows); local_summaries[level] = summary
    pd.DataFrame(local_rows).to_csv(AUDIT_ROOT / "local_probe_metrics.csv", index=False)
    invariance = {level: photometric_invariance(validation_features, level) for level in ("L0", "L1")}
    best_local = max(local_summaries, key=lambda level: local_summaries[level]["control_delta_ci95_low"])
    best_invariance = max(invariance, key=lambda level: invariance[level]["margin_ci95_low"])
    local_pass = (local_summaries[best_local]["control_delta_ci95_low"] > 0 and
                  invariance[best_invariance]["margin_ci95_low"] > 0)

    global_summary = retrieval_evidence(validation_features)
    global_pass = (global_summary["permutation_p_value"] < 0.05 and
                   global_summary["paired_margin_ci95_low"] > 0 and
                   global_summary["recall_at_1"] > 1.0 / global_summary["n_pairs"])
    statistics = feature_statistics(validation_features)
    if any(not np.isfinite(v["aggregate_std"]) or v["aggregate_std"] <= FEATURE_STD_MIN
           for v in statistics.values()):
        raise RuntimeError("structural feature-collapse gate failed")
    save_local_visuals(validation_features, probabilities, targets)

    after_hash = tensor_state_sha256(encoder.state_dict())
    if before_hash != after_hash or any(parameter.grad is not None for parameter in encoder.parameters()):
        raise RuntimeError("evaluation probe mutated the frozen encoder")
    preservation = p2_preservation(local_summaries, global_summary)
    comparison = stage_comparison(local_summaries, global_summary)
    write_json(AUDIT_ROOT / "clean_feature_manifest.json",
               {"schema_version": AUDIT_SCHEMA, "stage": AUDIT_STAGE, "fold": FOLD,
                "records": feature_records})
    config = {
        "schema_version": AUDIT_SCHEMA, "stage": AUDIT_STAGE, "fold": FOLD, "seed": SEED,
        "manifest_sha256": manifest_meta["sha256"], "checkpoint_path": str(checkpoint_path),
        "checkpoint_sha256": sha256_file(checkpoint_path), "encoder_state_sha256": before_hash,
        "train_sample_ids": sorted(train_rows.sample_id.tolist()),
        "validation_sample_ids": sorted(validation_rows.sample_id.tolist()),
        "test_sample_ids_not_opened": sorted(test_rows.sample_id.tolist()),
        "projection_geometry": PROJECTION_GEOMETRY,
        "target_use": "evaluation_only_silhouettes_never_used_for_fcmae_training_or_selection",
        "probe": {"kind": "1x1_conv2d", "epochs": PROBE_EPOCHS, "batch": PROBE_BATCH,
                  "optimizer": "AdamW", "lr": PROBE_LR, "weight_decay": PROBE_WEIGHT_DECAY,
                  "loss": "0.5_bce+0.5_soft_dice", "threshold": 0.5},
        "bootstrap_draws": BOOTSTRAPS, "retrieval_permutations": PERMUTATIONS,
        "software": {"python": platform.python_version(), "torch": torch.__version__,
                     "numpy": np.__version__}, "device": str(DEVICE),
    }
    write_json(AUDIT_ROOT / "config.json", config)
    warnings = []
    if comparison.get("status") == "INCOMPLETE": warnings.append("reference_comparison_incomplete")
    pinned_delta = comparison.get("comparisons", {}).get("pinned_init", {}).get("local_macro_dice_change")
    if pinned_delta is not None and pinned_delta < -LOCAL_REGRESSION_TOLERANCE:
        warnings.append("local_representation_regressed_from_pinned_initialization")
    if not local_pass: warnings.append("local_feature_evidence_below_threshold")
    if not global_pass: warnings.append("global_feature_evidence_below_threshold")
    if preservation["status"] == "WARN": warnings.append("p2_preservation_warning")
    summary = {
        "schema_version": AUDIT_SCHEMA, "stage": AUDIT_STAGE, "fold": FOLD,
        "structural_status": "PASS", "quality_status": "PASS" if not warnings else "WARN",
        "warnings": warnings,
        "local": {"status": "PASS" if local_pass else "WARN", "best_probe_level": best_local,
                  "best_invariance_level": best_invariance, "probes": local_summaries,
                  "photometric_invariance": invariance},
        "global": {"status": "PASS" if global_pass else "WARN", **global_summary},
        "feature_statistics": statistics, "stage_comparison": comparison, "p2_preservation": preservation,
        "encoder_unchanged": True, "test_paths_opened": False,
        "resource_usage": resource_usage(started),
        "qa_figures": sorted(str(path) for path in AUDIT_ROOT.glob("*.png")),
    }
    write_json(AUDIT_ROOT / "feature_audit_summary.json", summary)
    return summary


In [14]:
def synthetic_audit_contract_tests():
    mask = np.zeros((2, 4, 16, 16), dtype=np.float32); mask[:, :, 3:8, 4:10] = 1
    assert np.allclose(hard_dice_iou(mask, mask)[0], 1.0)
    empty = np.zeros_like(mask); assert np.allclose(hard_dice_iou(empty, empty)[1], 1.0)
    similarity = np.eye(4, dtype=np.float32)
    assert np.array_equal(retrieval_ranks(similarity, np.arange(4)), np.ones(4, dtype=int))
    assert stable_seed(SEED, FOLD, "audit") == stable_seed(SEED, FOLD, "audit")
    print("encoder feature-audit synthetic contract tests: PASS")


synthetic_audit_contract_tests()

if RUN_AUDIT:
    print(json.dumps(run_feature_audit(), indent=2))
else:
    print("DATA-FREE MODE: audit definitions loaded; no checkpoint, target, DRR, or test path was opened.")


encoder feature-audit synthetic contract tests: PASS
DATA-FREE MODE: audit definitions loaded; no checkpoint, target, DRR, or test path was opened.
